# 18 · Beyond classification: detection & segmentation

The contract is task-agnostic — only the **metric** and the **slice definitions**
change. Real detection/segmentation needs the BDD labels and a GPU; here we show
the metric swap with a tiny **worst-group IoU** computed on toy masks.

In [1]:
import sys, os, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
sys.path.insert(0, str(Path.cwd().parent))
import matplotlib; matplotlib.use("Agg")
import numpy as np, matplotlib.pyplot as plt
from harness.data import make_synthetic_dataset, DatasetSpec, load_split, class_names

DATA = Path("../data/bdd-tiny.lance")
if not DATA.exists():
    make_synthetic_dataset(DATA, DatasetSpec(n=3000, seed=7))
print("dataset:", DATA, "| NOTE: these chapters scale to bdd-small/full; here we")
print("demonstrate the mechanics on the tiny tier so they run with no GPU/cluster.")

dataset: ../data/bdd-tiny.lance | NOTE: these chapters scale to bdd-small/full; here we
demonstrate the mechanics on the tiny tier so they run with no GPU/cluster.


In [2]:
import numpy as np
rng = np.random.default_rng(0)
# toy: 200 frames, per-frame drivable-area mask IoU, tagged by weather
weather = rng.choice(["clear", "rainy", "night_rain"], size=200, p=[0.5, 0.3, 0.2])
quality = {"clear": 0.85, "rainy": 0.7, "night_rain": 0.5}   # night-rain is worst
iou = np.clip([rng.normal(quality[w], 0.08) for w in weather], 0, 1)

def worst_group_miou(iou, groups, macro_w=0.25):
    per = {g: float(iou[groups == g].mean()) for g in sorted(set(groups))}
    worst = min(per.values()); macro = float(np.mean(list(per.values())))
    return {"score": round(worst + macro_w * macro, 4), "per_group": {k: round(v,3) for k,v in per.items()},
            "worst": min(per, key=per.get)}
print(worst_group_miou(iou, weather))

{'score': 0.6438, 'per_group': {np.str_('clear'): 0.845, np.str_('night_rain'): 0.475, np.str_('rainy'): 0.706}, 'worst': np.str_('night_rain')}


Swap `worst_group_accuracy` for `worst_group_mIoU` (segmentation) or
`worst_group_mAP` (detection) and the rest of the harness — budget, mining, loop,
guardrails — is unchanged. The safety-critical worst slice (here night+rain) is
exactly what average mIoU would hide, which is why the objective targets it.